# 🔧 Amazon Reviews — Preprocessing Pipeline

This notebook takes the raw `reviews.csv` and produces:
- **Three raw splits** saved to disk: `data/train/`, `data/val/`, `data/test/` (80/10/10, stratified)
- **A fitted preprocessor** `PipelineModel` saved to `data/models/preprocessor/`
- **Three featurized splits** saved to `data/train_feat/`, `data/val_feat/`, `data/test_feat/`

### Pipeline Stages
| # | Stage | Input → Output |
|---|---|---|
| 1 | `RegexTokenizer` | `Text` → `words` |
| 2 | `StopWordsRemover` (negation-aware) | `words` → `filtered` |
| 3 | `LemmatizerTransformer` (custom) | `filtered` → `lemmatized` |
| 4 | `NGram(n=2)` | `lemmatized` → `bigrams` |
| 5 | `array_union` UDF | `lemmatized` + `bigrams` → `tokens` |
| 6 | `HashingTF(numFeatures=100_000)` | `tokens` → `raw_features` |
| 7 | `IDF` | `raw_features` → `features` |

> **Leakage rule:** The preprocessor is **fit on train only**, then applied to val and test.


---
## 0. Setup & SparkSession

In [1]:
import os
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, udf, array_union, rand
from pyspark.sql.types import ArrayType, StringType
from pyspark.ml import Pipeline, PipelineModel, Transformer
from pyspark.ml.param.shared import Param, Params
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, NGram
from nltk.stem import WordNetLemmatizer

# ── Paths ────────────────────────────────────────────────────────
DATA_DIR         = '../data'
INPUT_FILE       = os.path.join(DATA_DIR, 'reviews.csv')
TRAIN_DIR        = os.path.join(DATA_DIR, 'train')
VAL_DIR          = os.path.join(DATA_DIR, 'val')
TEST_DIR         = os.path.join(DATA_DIR, 'test')
TRAIN_FEAT_DIR   = os.path.join(DATA_DIR, 'train_feat')
VAL_FEAT_DIR     = os.path.join(DATA_DIR, 'val_feat')
TEST_FEAT_DIR    = os.path.join(DATA_DIR, 'test_feat')
PREPROCESSOR_DIR = os.path.join(DATA_DIR, 'models', 'preprocessor')

# ── Config ───────────────────────────────────────────────────────
NUM_FEATURES = 100_000   # was 10_000 — reduces hash collisions significantly
RANDOM_SEED  = 42

# ── SparkSession ─────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName('AmazonReviews-Preprocessing')
    .master('local[*]')
    .config('spark.driver.memory', '6g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')


26/05/05 18:40:43 WARN Utils: Your hostname, regisx001 resolves to a loopback address: 127.0.1.1; using 172.17.0.1 instead (on interface docker0)
26/05/05 18:40:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 18:40:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1 ready.


26/05/05 18:40:54 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


---
## 1. Load & Clean Raw Data

In [16]:
print('Loading reviews.csv...')
df = spark.read.csv(INPUT_FILE, header=True, inferSchema=True)
raw_count = df.count()
print(f'Raw rows: {raw_count:,}')
df.printSchema()


Loading reviews.csv...
Raw rows: 568,454
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: string (nullable = true)
 |-- HelpfulnessDenominator: string (nullable = true)
 |-- Score: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)



In [18]:
# ── Step 1: Drop rows with null or empty Text ────────────────────
df = df.na.drop(subset=['Text'])
df = df.withColumn('Text', col('Text').cast('string'))
df = df.filter(col('Text') != '')

# ── Step 2: Drop duplicate Text entries (prevents train/test leakage) ──
df = df.dropDuplicates(['Text'])

# ── Step 3: Remove invalid helpfulness rows ───────────────────────
df = df.filter(
    (col('HelpfulnessDenominator') == 0) |
    (col('HelpfulnessNumerator') <= col('HelpfulnessDenominator'))
)

# ── Step 4: Cast Summary to string, fill nulls ───────────────────
df = df.withColumn('Summary', col('Summary').cast('string'))
df = df.fillna({'Summary': ''})

clean_count = df.count()
print(f'After cleaning: {clean_count:,} rows  (removed {raw_count - clean_count:,})')


After cleaning: 388,301 rows  (removed 180,153)


---
## 2. Create Target Label (Sentiment)

In [19]:
df = df.withColumn(
    'Sentiment',
    when(col('Score') < 3, 'negative')
    .when(col('Score') == 3, 'neutral')
    .otherwise('positive')
)

# Numeric label for ML: negative=0, neutral=1, positive=2
df = df.withColumn(
    'label',
    when(col('Sentiment') == 'negative', 0)
    .when(col('Sentiment') == 'neutral',  1)
    .otherwise(2)
)

print('Sentiment distribution:')
df.groupBy('Sentiment', 'label').count().orderBy('label').show()


Sentiment distribution:


+---------+-----+------+
|Sentiment|label| count|
+---------+-----+------+
| negative|    0| 55222|
|  neutral|    1| 29127|
| positive|    2|303952|
+---------+-----+------+



---
## 3. Stratified 80 / 10 / 10 Split

We split **per class** to preserve the class ratio across train, val, and test.  
This is critical given the severe class imbalance (~78% positive, ~7% neutral).


In [20]:
def stratified_split(df, label_col='Sentiment', train_ratio=0.8,
                     val_ratio=0.1, seed=42):
    """
    Split a Spark DataFrame per class to preserve class ratios.
    Returns (train_df, val_df, test_df).
    """
    train_parts, val_parts, test_parts = [], [], []
    classes = [r[label_col] for r in df.select(label_col).distinct().collect()]

    for cls in classes:
        subset = df.filter(col(label_col) == cls)
        # First split: train vs rest
        rest_ratio = 1.0 - train_ratio
        t, rest = subset.randomSplit([train_ratio, rest_ratio], seed=seed)
        # Second split: val vs test (50/50 of the remaining 20%)
        v, te = rest.randomSplit([0.5, 0.5], seed=seed)
        train_parts.append(t)
        val_parts.append(v)
        test_parts.append(te)

    from functools import reduce
    train_df = reduce(lambda a, b: a.union(b), train_parts)
    val_df   = reduce(lambda a, b: a.union(b), val_parts)
    test_df  = reduce(lambda a, b: a.union(b), test_parts)
    return train_df, val_df, test_df


train_df, val_df, test_df = stratified_split(df, seed=RANDOM_SEED)

# Shuffle to avoid class ordering artefacts
train_df = train_df.orderBy(rand(seed=RANDOM_SEED))
val_df   = val_df.orderBy(rand(seed=RANDOM_SEED))
test_df  = test_df.orderBy(rand(seed=RANDOM_SEED))

print(f'Train : {train_df.count():>7,}')
print(f'Val   : {val_df.count():>7,}')
print(f'Test  : {test_df.count():>7,}')


Train : 310,886


Val   :  39,023


Test  :  38,392


In [21]:
# Verify class distribution is preserved in each split
print('--- Train distribution ---')
train_df.groupBy('Sentiment').count().orderBy('count', ascending=False).show()

print('--- Val distribution ---')
val_df.groupBy('Sentiment').count().orderBy('count', ascending=False).show()

print('--- Test distribution ---')
test_df.groupBy('Sentiment').count().orderBy('count', ascending=False).show()


--- Train distribution ---


+---------+------+
|Sentiment| count|
+---------+------+
| positive|243060|
| negative| 44350|
|  neutral| 23476|
+---------+------+

--- Val distribution ---


+---------+-----+
|Sentiment|count|
+---------+-----+
| positive|30661|
| negative| 5509|
|  neutral| 2853|
+---------+-----+

--- Test distribution ---


+---------+-----+
|Sentiment|count|
+---------+-----+
| positive|30231|
| negative| 5363|
|  neutral| 2798|
+---------+-----+



---
## 4. Save Raw Splits

Saving the raw splits **before** featurization so future experiments can re-run the
preprocessing pipeline with different parameters without changing the 80/10/10 boundary.


In [22]:
train_df.coalesce(1).write.csv(TRAIN_DIR, mode='overwrite', header=True)
val_df.coalesce(1).write.csv(VAL_DIR,   mode='overwrite', header=True)
test_df.coalesce(1).write.csv(TEST_DIR,  mode='overwrite', header=True)

print('Raw splits saved:')
print(f'   {TRAIN_DIR}')
print(f'   {VAL_DIR}')
print(f'   {TEST_DIR}')


Raw splits saved:
   ../data/train
   ../data/val
   ../data/test


---
## 5. Custom NLP Components

### Why a custom `LemmatizerTransformer`?
PySpark's MLlib has no native lemmatizer. We wrap NLTK's `WordNetLemmatizer` in a custom
`Transformer` subclass so it integrates cleanly into a `Pipeline` and can be saved/loaded
alongside the other stages.

### Verb-first lemmatization strategy
We try the **verb** form first (`'loved' → 'love'`), then the **noun** form (`'batteries' → 'battery'`).
Only fall back to lowercased original if neither lemmatization changes the token.

### Negation-aware stop word removal
The default English stop word list removes 'not', 'no', 'never' — which are critical for
negative sentiment. We explicitly **keep negations** in the stop word list.


In [23]:
# ── Negation-aware stop word list ────────────────────────────────
NEGATION_WORDS = {
    'not', 'no', 'nor', 'never', 'neither', 'none', 'nobody',
    'nowhere', 'hardly', 'barely', 'scarcely'
}
default_stopwords  = StopWordsRemover.loadDefaultStopWords('english')
custom_stopwords   = [w for w in default_stopwords if w not in NEGATION_WORDS]

print(f'Default stop words : {len(default_stopwords)}')
print(f'Custom stop words  : {len(custom_stopwords)}  (kept {len(NEGATION_WORDS)} negations)')
print(f'Preserved negations: {NEGATION_WORDS}')


Default stop words : 181
Custom stop words  : 178  (kept 11 negations)
Preserved negations: {'hardly', 'none', 'barely', 'nobody', 'scarcely', 'never', 'nowhere', 'nor', 'no', 'not', 'neither'}


In [24]:
# ── Lemmatization function (used by custom Transformer + UDF) ────
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    """Verb-first lemmatization with NLTK WordNetLemmatizer."""
    if tokens is None:
        return []
    result = []
    for w in tokens:
        lemma_v = lemmatizer.lemmatize(w, pos='v')   # verb form first
        lemma_n = lemmatizer.lemmatize(w, pos='n')   # then noun
        if lemma_v != w:
            result.append(lemma_v)
        elif lemma_n != w:
            result.append(lemma_n)
        else:
            result.append(w.lower())
    return result

lemmatize_udf = udf(lemmatize_tokens, ArrayType(StringType()))
print('Lemmatization UDF ready.')


Lemmatization UDF ready.


In [26]:
# ── Custom LemmatizerTransformer ──────────────────────────────────
class LemmatizerTransformer(Transformer):
    """
    PySpark Pipeline-compatible Transformer that lemmatizes
    an ArrayType(StringType) column using NLTK WordNetLemmatizer.
    Verb-first strategy: tries verb lemma, then noun, else lowercase.
    """
    inputCol  = Param(Params._dummy(), 'inputCol',  'Input column name (ArrayType)')
    outputCol = Param(Params._dummy(), 'outputCol', 'Output column name (ArrayType)')

    def __init__(self, inputCol='filtered', outputCol='lemmatized'):
        super().__init__()
        self._setDefault(inputCol='filtered', outputCol='lemmatized')
        self._set(inputCol=inputCol, outputCol=outputCol)

    def setInputCol(self, v):  return self._set(inputCol=v)
    def setOutputCol(self, v): return self._set(outputCol=v)

    def _transform(self, dataset):
        in_col  = self.getOrDefault(self.inputCol)
        out_col = self.getOrDefault(self.outputCol)
        _udf    = udf(lemmatize_tokens, ArrayType(StringType()))
        return dataset.withColumn(out_col, _udf(col(in_col)))

print('LemmatizerTransformer class defined.')


LemmatizerTransformer class defined.


---
## 6. Build & Fit Preprocessing Pipeline

### Why bigrams?
Unigrams alone lose context: 'not' and 'good' are two separate tokens.
**Bigrams** capture 'not_good', 'very_bad', 'highly_recommend' as single features,
improving recall on the negative and neutral classes.

We combine unigrams + bigrams into one token list before hashing,
giving the model access to both word-level and phrase-level signals.


In [ ]:
# ── Stage 1: Regex Tokenizer ──────────────────────────────────────
# Splits on \w+ (word characters), skipping punctuation and numbers-only tokens
tokenizer = RegexTokenizer(
    inputCol='Text', outputCol='words',
    pattern=r'\w+', gaps=False,
    minTokenLength=2          # skip single-character tokens
)

# ── Stage 2: Negation-aware Stop Word Remover ─────────────────────
remover = StopWordsRemover(
    inputCol='words', outputCol='filtered',
    stopWords=custom_stopwords
)

# ── Stage 3: Lemmatizer ───────────────────────────────────────────
lemmatizer_t = LemmatizerTransformer(
    inputCol='filtered', outputCol='lemmatized'
)

# ── Stage 4: Bigram extractor ─────────────────────────────────────
bigram = NGram(n=2, inputCol='lemmatized', outputCol='bigrams')

# ── Stage 5: Combine unigrams + bigrams ───────────────────────────
# We use a UDF because PySpark has no native array_union pipeline stage
combine_udf = udf(lambda a, b: (a or []) + (b or []), ArrayType(StringType()))

from pyspark.ml import Transformer

class UnigramBigramCombiner(Transformer):
    """
    Concatenates two ArrayType columns (unigrams + bigrams) into one.
    """
    unigram_col = Param(Params._dummy(), 'unigram_col', 'unigram column')
    bigram_col  = Param(Params._dummy(), 'bigram_col',  'bigram column')
    outputCol   = Param(Params._dummy(), 'outputCol',   'output column')

    def __init__(self, unigram_col='lemmatized', bigram_col='bigrams', outputCol='tokens'):
        super().__init__()
        self._setDefault(unigram_col='lemmatized', bigram_col='bigrams', outputCol='tokens')
        self._set(unigram_col=unigram_col, bigram_col=bigram_col, outputCol=outputCol)

    def _transform(self, dataset):
        uc  = self.getOrDefault(self.unigram_col)
        bc  = self.getOrDefault(self.bigram_col)
        oc  = self.getOrDefault(self.outputCol)
        _u  = udf(lambda a, b: (a or []) + (b or []), ArrayType(StringType()))
        return dataset.withColumn(oc, _u(col(uc), col(bc)))

combiner = UnigramBigramCombiner(
    unigram_col='lemmatized', bigram_col='bigrams', outputCol='tokens'
)

# ── Stage 6: HashingTF ────────────────────────────────────────────
hashing_tf = HashingTF(
    inputCol='tokens', outputCol='raw_features',
    numFeatures=NUM_FEATURES        # 100k reduces hash collisions vs 10k
)

# ── Stage 7: IDF ──────────────────────────────────────────────────
idf = IDF(inputCol='raw_features', outputCol='features', minDocFreq=3)

# ── Assemble Pipeline ─────────────────────────────────────────────
preprocessing_pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    lemmatizer_t,
    bigram,
    combiner,
    hashing_tf,
    idf
])

print('Preprocessing pipeline assembled.')
print(f'   Stages: {[type(s).__name__ for s in preprocessing_pipeline.getStages()]}')


Preprocessing pipeline assembled.
   Stages: ['RegexTokenizer', 'StopWordsRemover', 'LemmatizerTransformer', 'NGram', 'UnigramBigramCombiner', 'HashingTF', 'IDF']


In [29]:
# ── FIT on TRAIN only — prevent leakage ──────────────────────────
print('Fitting preprocessor on train set (this may take a few minutes)...')
preprocessor = preprocessing_pipeline.fit(train_df)
print('Preprocessor fitted.')


Fitting preprocessor on train set (this may take a few minutes)...


✅Preprocessor fitted.


In [30]:
# ── Transform all three splits ────────────────────────────────────
print('Transforming splits...')
train_feat = preprocessor.transform(train_df)
val_feat   = preprocessor.transform(val_df)
test_feat  = preprocessor.transform(test_df)

# Keep only the columns needed for training
KEEP_COLS = ['Id', 'ProductId', 'UserId', 'Score', 'Time',
             'Sentiment', 'label', 'features']
train_feat = train_feat.select(KEEP_COLS)
val_feat   = val_feat.select(KEEP_COLS)
test_feat  = test_feat.select(KEEP_COLS)

print('All splits transformed.')
train_feat.printSchema()


Transforming splits...
All splits transformed.
root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- Score: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Sentiment: string (nullable = false)
 |-- label: integer (nullable = false)
 |-- features: vector (nullable = true)



---
## 7. Verify Pipeline Output

In [31]:
# Quick sanity checks
print('=== Feature vector sample (train) ===')
train_feat.select('Sentiment', 'label', 'features').show(5, truncate=80)

print('\n=== Feature vector sparsity ===')
from pyspark.sql.functions import size
# Count non-zero elements using VectorUDT
from pyspark.ml.functions import vector_to_array
train_feat.withColumn(
    'nnz', size(vector_to_array('features').cast('array<double>'))
).select('nnz').summary('min','mean','max').show()


=== Feature vector sample (train) ===


26/05/05 18:51:07 WARN DAGScheduler: Broadcasting large task binary with size 1731.6 KiB


+---------+-----+--------------------------------------------------------------------------------+
|Sentiment|label|                                                                        features|
+---------+-----+--------------------------------------------------------------------------------+
| positive|    2|(100000,[1055,1622,4271,13525,16717,17405,18110,18383,22894,23506,33420,34294...|
|  neutral|    1|(100000,[1810,5419,6035,6531,8080,10080,15815,22886,26246,26567,26657,29693,3...|
| positive|    2|(100000,[2543,5702,5734,6493,7156,8284,10015,10157,10281,10288,11288,13525,14...|
| positive|    2|(100000,[2972,4219,4549,5149,6290,8149,10157,10447,10475,10973,11226,12242,13...|
| positive|    2|(100000,[6664,8494,10157,10763,11226,11623,12007,12180,18605,19848,22262,2988...|
+---------+-----+--------------------------------------------------------------------------------+
only showing top 5 rows


=== Feature vector sparsity ===


26/05/05 18:51:11 WARN DAGScheduler: Broadcasting large task binary with size 1761.8 KiB


+-------+--------+
|summary|     nnz|
+-------+--------+
|    min|  100000|
|   mean|100000.0|
|    max|  100000|
+-------+--------+



In [32]:
# Verify class distribution is preserved after transformation
print('=== Label distribution in train_feat ===')
train_feat.groupBy('Sentiment', 'label').count().orderBy('label').show()


=== Label distribution in train_feat ===


+---------+-----+------+
|Sentiment|label| count|
+---------+-----+------+
| negative|    0| 44350|
|  neutral|    1| 23476|
| positive|    2|243060|
+---------+-----+------+



---
## 8. Compute Class Weights

To address the **~10:1 positive:neutral imbalance**, we compute inverse-frequency
class weights. These will be passed to the classifier in the training notebook
via a `weightCol` column.

Formula: `weight(class_i) = total_samples / (n_classes × count(class_i))`


In [33]:
from pyspark.sql.functions import count as spark_count

n_total   = train_feat.count()
n_classes = 3
class_counts = (
    train_feat.groupBy('label')
    .agg(spark_count('label').alias('cnt'))
    .orderBy('label')
    .collect()
)

weight_map = {}
print('=== Class Weights (inverse frequency) ===')
for row in class_counts:
    w = n_total / (n_classes * row['cnt'])
    weight_map[row['label']] = w
    label_name = {0: 'negative', 1: 'neutral', 2: 'positive'}[row['label']]
    print(f'  label={row["label"]} ({label_name:>8s}): count={row["cnt"]:>7,}  weight={w:.4f}')

# Add weight column to all splits
def add_weights(df, wmap):
    expr = None
    for lbl, w in wmap.items():
        cond = when(col('label') == lbl, float(w))
        expr = cond if expr is None else expr.when(col('label') == lbl, float(w))
    return df.withColumn('classWeight', expr.otherwise(1.0))

train_feat = add_weights(train_feat, weight_map)
val_feat   = add_weights(val_feat,   weight_map)
test_feat  = add_weights(test_feat,  weight_map)

print('\n column added to all splits.')


=== Class Weights (inverse frequency) ===
  label=0 (negative): count= 44,350  weight=2.3366
  label=1 ( neutral): count= 23,476  weight=4.4142
  label=2 (positive): count=243,060  weight=0.4264

 column added to all splits.


---
## 9. Save Featurized Splits & Preprocessor

In [34]:
# Save featurized splits as Parquet (preserves vector type)
print('Saving featurized splits...')
train_feat.write.parquet(TRAIN_FEAT_DIR, mode='overwrite')
val_feat.write.parquet(VAL_FEAT_DIR,     mode='overwrite')
test_feat.write.parquet(TEST_FEAT_DIR,   mode='overwrite')

print('Featurized splits saved:')
print(f'   {TRAIN_FEAT_DIR}')
print(f'   {VAL_FEAT_DIR}')
print(f'   {TEST_FEAT_DIR}')


Saving featurized splits...


26/05/05 18:53:55 WARN DAGScheduler: Broadcasting large task binary with size 1943.8 KiB
26/05/05 18:55:03 WARN DAGScheduler: Broadcasting large task binary with size 1949.4 KiB
26/05/05 18:55:31 WARN DAGScheduler: Broadcasting large task binary with size 1949.4 KiB


Featurized splits saved:
   ../data/train_feat
   ../data/val_feat
   ../data/test_feat


In [ ]:
# ── Save only the FITTED (stateful) stages ────────────────────────────────
# LemmatizerTransformer and UnigramBigramCombiner are STATELESS — they have
# no learned parameters, so there's nothing to save/load for them.
# Only HashingTF and IDF are fitted (learned from training data).
#
# We save them individually. The training + streaming notebooks will
# re-create the stateless stages and load these two fitted models.


fitted_hashing_tf = preprocessor.stages[5]   # HashingTFModel
fitted_idf        = preprocessor.stages[6]   # IDFModel


HASHING_TF_DIR = os.path.join(DATA_DIR, 'models', 'hashing_tf')
IDF_DIR        = os.path.join(DATA_DIR, 'models', 'idf')


fitted_hashing_tf.write().overwrite().save(HASHING_TF_DIR)
print(f' HashingTF model saved → {HASHING_TF_DIR}')


fitted_idf.write().overwrite().save(IDF_DIR)
print(f' IDF model saved       → {IDF_DIR}')


# ── How to reload in other notebooks ─────────────────────────────────────
# from pyspark.ml.feature import HashingTFModel, IDFModel
#
# fitted_hashing_tf = HashingTFModel.load('../data/models/hashing_tf')
# fitted_idf        = IDFModel.load('../data/models/idf')
#
# Then re-apply the stateless stages manually before calling these models.

 HashingTF model saved → ../data/models/hashing_tf
 IDF model saved       → ../data/models/idf


26/05/05 18:58:07 WARN TaskSetManager: Stage 272 contains a task of very large size (1602 KiB). The maximum recommended task size is 1000 KiB.


---
## 10. Summary

| Artifact | Path |
|---|---|
| Raw train split | `data/train/` |
| Raw val split | `data/val/` |
| Raw test split | `data/test/` |
| Featurized train | `data/train_feat/` |
| Featurized val | `data/val_feat/` |
| Featurized test | `data/test_feat/` |
| HashingTF model | `data/models/hashing_tf/` |
| IDF model | `data/models/idf/` |

### Key design choices
- **`numFeatures=100_000`** — 10× more than previous version, reduces hash collisions
- **Bigrams** — captures 'not good', 'very bad', 'highly recommend'
- **`minDocFreq=3`** on IDF — filters noise tokens appearing in fewer than 3 documents
- **Fit on train only** — no leakage into val/test
- **`classWeight` column** — ready for weighted training in the next notebook

### ➡ Next → `03-train-model.ipynb`
Load `data/train_feat/` and `data/val_feat/`, train multiple classifiers with
class weights, compare F1 scores, tune hyperparameters on val set.


In [ ]:
spark.stop()
print('SparkSession stopped.')
